In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3
from langchain_google_genai import ChatGoogleGenerativeAI
from dotenv import load_dotenv

c:\rachna\test\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
# Load environment variables and initialize the language model
load_dotenv() 
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite")

In [3]:
# Define the state structure for the order workflow
class OrderState(TypedDict):
    order_id: str
    item: str
    quantity: int
    in_stock: bool
    status: str
    customer_message: str

In [4]:
# Define node functions
def validate_order(state: OrderState):
    print("Validating order...")
    return {}

def check_inventory(state: OrderState):
    print("Checking inventory...")

    if state["quantity"] <= 2: 
        return {"in_stock": True}
    else:
        return {"in_stock": False}

def fulfill_order(state: OrderState):
    print("Deciding fulfillment...")

    if state["in_stock"]:
        return {"status": "ORDER_CONFIRMED"}
    else:
        return {"status": "OUT_OF_STOCK"}
    
def generate_message(state: OrderState):
    print("Generating customer message ")

    prompt = f"""
    You are an e-commerce assistant.
    Order ID: {state['order_id']}
    Item: {state['item']}
    Status: {state['status']}
    Write a short, friendly message to the customer.
    """
    response = llm.invoke(prompt).content
    return {"customer_message": response}


In [5]:
# Build the graph
graph_builder = StateGraph(OrderState)

graph_builder.add_node("validate_order", validate_order)
graph_builder.add_node("check_inventory", check_inventory)
graph_builder.add_node("fulfill_order", fulfill_order)
graph_builder.add_node("generate_message", generate_message)

graph_builder.add_edge(START, "validate_order")
graph_builder.add_edge("validate_order", "check_inventory")
graph_builder.add_edge("check_inventory", "fulfill_order")
graph_builder.add_edge("fulfill_order", "generate_message")
graph_builder.add_edge("generate_message", END)

In [6]:
# Configure SQLite-based checkpoint persistence and compile the graph
conn = sqlite3.connect("order-checkpoints.sqlite", check_same_thread=False)
checkpointer = SqliteSaver(conn)
graph = graph_builder.compile(checkpointer=checkpointer)

In [7]:
# Execute the graph with an initial order input
config = {"configurable": {"thread_id": "order-001"}}

graph.invoke(
    {
        "order_id": "ORD-101",
        "item": "Wireless Mouse",
        "quantity": 5
    },
    config=config
)


Validating order...
Checking inventory...
Deciding fulfillment...
Generating customer message 


{'order_id': 'ORD-101',
 'item': 'Wireless Mouse',
 'quantity': 5,
 'in_stock': False,
 'status': 'OUT_OF_STOCK',
 'customer_message': "Subject: Quick Update on Your ORD-101 Order\n\nHi there!\n\nWe're writing to you about your recent order, ORD-101, for the Wireless Mouse.\n\nUnfortunately, this item is currently out of stock. We know this is disappointing, and we sincerely apologize for any inconvenience this may cause.\n\nWe're working hard to get it back in stock as soon as possible. We'll send you another update as soon as it's available again.\n\nThanks for your patience and understanding!\n\nBest regards,\n\nYour E-commerce Assistant"}

In [8]:
# Retrieve and inspect all saved checkpoints for the thread
for checkpoint in graph.get_state_history({"configurable": {"thread_id": "order-001"}}):
    print(checkpoint.config["configurable"]["checkpoint_id"])
    print(checkpoint.next)
    print()


1f125de4-cad7-6518-8004-0b31e5628ae2
()

1f125de4-bf4a-61c2-8003-66610f24abba
('generate_message',)

1f125de4-bf45-6766-8002-2a85b14b5d3b
('fulfill_order',)

1f125de4-bf3d-6ef9-8001-df332f0522de
('check_inventory',)

1f125de4-bf34-6131-8000-0361fa8c25cd
('validate_order',)

1f125de4-bf2a-6503-bfff-0bffda86bb2f
('__start__',)



In [9]:
# Resume graph execution from a specific checkpoint
graph.invoke(None, config={"configurable": {"thread_id": "order-001", "checkpoint_id": "1f125de4-bf45-6766-8002-2a85b14b5d3b"}})


Deciding fulfillment...
Generating customer message 


{'order_id': 'ORD-101',
 'item': 'Wireless Mouse',
 'quantity': 5,
 'in_stock': False,
 'status': 'OUT_OF_STOCK',
 'customer_message': "Subject: An Update on Your Order ORD-101\n\nHi there,\n\nThank you for your recent order of the Wireless Mouse (ORD-101)!\n\nWe're writing to let you know that unfortunately, this item is currently out of stock. We're working hard to get it back on our shelves as soon as possible.\n\nWe'll send you another update as soon as it's available again. In the meantime, please let us know if you have any questions or if you'd like to explore alternative options.\n\nThanks for your patience!\n\nSincerely,\nYour E-commerce Team"}

In [10]:
# Inspect checkpoint history after resuming execution
for checkpoint in graph.get_state_history({"configurable": {"thread_id": "order-001"}}):
    print(checkpoint.config["configurable"]["checkpoint_id"])
    print(checkpoint.next)
    print()

1f125de7-195b-62d2-8004-ec9dfb849a0a
()

1f125de7-0cef-6daa-8003-d2dcdcc08cb9
('generate_message',)

1f125de4-cad7-6518-8004-0b31e5628ae2
()

1f125de4-bf4a-61c2-8003-66610f24abba
('generate_message',)

1f125de4-bf45-6766-8002-2a85b14b5d3b
('fulfill_order',)

1f125de4-bf3d-6ef9-8001-df332f0522de
('check_inventory',)

1f125de4-bf34-6131-8000-0361fa8c25cd
('validate_order',)

1f125de4-bf2a-6503-bfff-0bffda86bb2f
('__start__',)



In [11]:
# Update the graph state at a specific checkpoint
new_config = graph.update_state(
    values={"in_stock": True},
    config={
        "configurable": {
            "thread_id": "order-001",
            "checkpoint_id": "1f125de4-bf45-6766-8002-2a85b14b5d3b",
            "checkpoint_ns": ""
        }
    }
)


In [12]:
# Inspect checkpoint history and state values after updating
for checkpoint in graph.get_state_history({"configurable": {"thread_id": "order-001"}}):
    print(checkpoint.config["configurable"]["checkpoint_id"])
    print(checkpoint.next)
    print(checkpoint.values)
    print()

1f125de8-fdc3-653a-8003-e339fd6cee95
('fulfill_order',)
{'order_id': 'ORD-101', 'item': 'Wireless Mouse', 'quantity': 5, 'in_stock': True}

1f125de7-195b-62d2-8004-ec9dfb849a0a
()
{'order_id': 'ORD-101', 'item': 'Wireless Mouse', 'quantity': 5, 'in_stock': False, 'status': 'OUT_OF_STOCK', 'customer_message': "Subject: An Update on Your Order ORD-101\n\nHi there,\n\nThank you for your recent order of the Wireless Mouse (ORD-101)!\n\nWe're writing to let you know that unfortunately, this item is currently out of stock. We're working hard to get it back on our shelves as soon as possible.\n\nWe'll send you another update as soon as it's available again. In the meantime, please let us know if you have any questions or if you'd like to explore alternative options.\n\nThanks for your patience!\n\nSincerely,\nYour E-commerce Team"}

1f125de7-0cef-6daa-8003-d2dcdcc08cb9
('generate_message',)
{'order_id': 'ORD-101', 'item': 'Wireless Mouse', 'quantity': 5, 'in_stock': False, 'status': 'OUT_OF_S

In [13]:
# Continue execution using the updated state configuration
graph.invoke(None, new_config)

Deciding fulfillment...
Generating customer message 


{'order_id': 'ORD-101',
 'item': 'Wireless Mouse',
 'quantity': 5,
 'in_stock': True,
 'status': 'ORDER_CONFIRMED',
 'customer_message': "Hi there!\n\nJust wanted to let you know that your order, ORD-101, for the Wireless Mouse has been confirmed! We'll get it ready for you soon.\n\nThanks for shopping with us!"}

In [14]:
# Final checkpoint history after completing resumed execution
for checkpoint in graph.get_state_history({"configurable": {"thread_id": "order-001"}}):
    print(checkpoint.config["configurable"]["checkpoint_id"])
    print(checkpoint.next)
    print()  

1f125dea-a9ca-67ca-8005-619735f9f569
()

1f125dea-9da1-6076-8004-19ed7d9cc853
('generate_message',)

1f125de8-fdc3-653a-8003-e339fd6cee95
('fulfill_order',)

1f125de7-195b-62d2-8004-ec9dfb849a0a
()

1f125de7-0cef-6daa-8003-d2dcdcc08cb9
('generate_message',)

1f125de4-cad7-6518-8004-0b31e5628ae2
()

1f125de4-bf4a-61c2-8003-66610f24abba
('generate_message',)

1f125de4-bf45-6766-8002-2a85b14b5d3b
('fulfill_order',)

1f125de4-bf3d-6ef9-8001-df332f0522de
('check_inventory',)

1f125de4-bf34-6131-8000-0361fa8c25cd
('validate_order',)

1f125de4-bf2a-6503-bfff-0bffda86bb2f
('__start__',)

